# Study 878 — Economic Policy Uncertainty — the teardown

The two predictive regressions (forward vol AND forward return on the uncertainty level/change) with Newey-West HAC *t*, the two-era cut, the block-shuffle placebo, the costed timer, and the live synthetic control. *Signal = labelled VIX proxy — the newspaper EPU feed was unreachable in-environment (see `docs/references.md`).*

In [1]:
R = {'source': 'vix_proxy', 'start': '1993-02-28', 'end': '2026-06-30', 'n': 401, 'fp': 'd922d69b4b52', 'unc_lo': 9.5, 'unc_hi': 59.9, 'unc_mean': 19.6, 'rv_lvl': {1: (10.21, 0.507), 3: (9.44, 0.388), 6: (7.41, 0.301), 12: (6.48, 0.231)}, 'rv_chg': {1: 3.39, 3: 3.09, 6: 2.49, 12: 2.73}, 'ret_lvl': {1: (0.65, 0.003), 3: (0.82, 0.01), 6: (1.24, 0.017), 12: (1.04, 0.01)}, 'ret_chg': {1: -0.49, 3: 0.36, 6: 0.86, 12: 0.26}, 'era_early_ret_t': 0.09, 'era_early_rv_t': 6.2, 'era_early_n': 191, 'era_late_ret_t': 4.36, 'era_late_rv_t': 8.18, 'era_late_n': 210, 'placebo_ret3_p': 0.258, 'placebo_ret6_p': 0.221, 'placebo_rv3_p': 0.0, 'timer_leanin_sharpe': 0.49, 'timer_leanin_ann': 5.9, 'timer_derisk_sharpe': 0.57, 'timer_derisk_ann': 5.2, 'timer_bh_sharpe': 0.77, 'timer_bh_ann': 11.4, 'null_ret_t': 0.1, 'null_rv_t': 1.12, 'planted_ret_t': 4.96, 'planted_rv_t': 14.26, 'null_fire': 1, 'null_seeds': 10}

## Leg 1 — forward realized vol on the uncertainty level (the 'vol story')

In [2]:
for h in (1,3,6,12):
    t,r2 = R['rv_lvl'][h]
    print(f"h={h:>2}m: HAC t = {t:+6.2f}   R2 = {r2:.3f}   (on change: t = {R['rv_chg'][h]:+.2f})")
print('\n-> strongly significant everywhere, BUT VIX *is* implied vol: this is the'
      ' near-mechanical variance-risk-premium fact, a coincident reading, not an edge.')

h= 1m: HAC t = +10.21   R2 = 0.507   (on change: t = +3.39)
h= 3m: HAC t =  +9.44   R2 = 0.388   (on change: t = +3.09)
h= 6m: HAC t =  +7.41   R2 = 0.301   (on change: t = +2.49)
h=12m: HAC t =  +6.48   R2 = 0.231   (on change: t = +2.73)

-> strongly significant everywhere, BUT VIX *is* implied vol: this is the near-mechanical variance-risk-premium fact, a coincident reading, not an edge.


## Leg 2 — forward SPY return on the uncertainty level (the 'risk-premium story')

In [3]:
for h in (1,3,6,12):
    t,r2 = R['ret_lvl'][h]
    print(f"h={h:>2}m: HAC t = {t:+6.2f}   R2 = {r2:.3f}   (on change: t = {R['ret_chg'][h]:+.2f})")
print('\n-> not one horizon clears |t|=2; R2 ~ 0.01. The risk-premium claim fails.')

h= 1m: HAC t =  +0.65   R2 = 0.003   (on change: t = -0.49)
h= 3m: HAC t =  +0.82   R2 = 0.010   (on change: t = +0.36)
h= 6m: HAC t =  +1.24   R2 = 0.017   (on change: t = +0.86)
h=12m: HAC t =  +1.04   R2 = 0.010   (on change: t = +0.26)

-> not one horizon clears |t|=2; R2 ~ 0.01. The risk-premium claim fails.


## Robustness — two eras (split 2009-01-01), horizon 3m

In [4]:
print(f"1993-2008 (n={R['era_early_n']}): RET t = {R['era_early_ret_t']:+.2f} | RV t = {R['era_early_rv_t']:+.2f}")
print(f"2009-2026 (n={R['era_late_n']}): RET t = {R['era_late_ret_t']:+.2f} | RV t = {R['era_late_rv_t']:+.2f}")
print('-> the return leg lives ONLY post-2009 (recovery drift) and is absent before: a single-era artefact.')

1993-2008 (n=191): RET t = +0.09 | RV t = +6.20
2009-2026 (n=210): RET t = +4.36 | RV t = +8.18
-> the return leg lives ONLY post-2009 (recovery drift) and is absent before: a single-era artefact.


## Placebo — block-shuffle the regressor (broken link), 1,000 draws

In [5]:
print(f"return h=3: p = {R['placebo_ret3_p']:.3f}   return h=6: p = {R['placebo_ret6_p']:.3f}")
print(f"vol    h=3: p = {R['placebo_rv3_p']:.3f}  (the vol slope is real; the return slope is not)")

return h=3: p = 0.258   return h=6: p = 0.221
vol    h=3: p = 0.000  (the vol slope is real; the return slope is not)


## The timer — lean INTO high uncertainty vs buy-and-hold

In [6]:
print(f"lean-in : ann {R['timer_leanin_ann']:+.1f}%  Sharpe {R['timer_leanin_sharpe']:.2f}")
print(f"de-risk : ann {R['timer_derisk_ann']:+.1f}%  Sharpe {R['timer_derisk_sharpe']:.2f}")
print(f"buy-hold: ann {R['timer_bh_ann']:+.1f}%  Sharpe {R['timer_bh_sharpe']:.2f}  <- neither rule beats it")

lean-in : ann +5.9%  Sharpe 0.49
de-risk : ann +5.2%  Sharpe 0.57
buy-hold: ann +11.4%  Sharpe 0.77  <- neither rule beats it


## Synthetic positive control — the machinery is unbiased (live)

The detector must recover planted forward relations on both legs and stay silent on the null.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from epu import data, strategy as st
null = st.synthetic_detect(*data.synthetic(360, 0.0, 0.0, 878), horizon=3)
planted = st.synthetic_detect(*data.synthetic(360, 0.02, 0.6, 878), horizon=3)
print(f"null    : ret_t {null['ret_t']:+.2f}  rv_t {null['rv_t']:+.2f}")
print(f"planted : ret_t {planted['ret_t']:+.2f}  rv_t {planted['rv_t']:+.2f}")
nr = np.array([st.synthetic_detect(*data.synthetic(300,0.0,0.0,878+s),3)['ret_t'] for s in range(10)])
print(f"null 10 seeds: ret_t mean {nr.mean():+.2f} (sd {nr.std(ddof=1):.2f}), |t|>=2 in {(abs(nr)>=2).sum()}/10")

null    : ret_t +0.10  rv_t +1.12
planted : ret_t +4.96  rv_t +14.26


null 10 seeds: ret_t mean -0.41 (sd 1.20), |t|>=2 in 1/10


## Verdict

- **Signal — None.** The risk-premium claim fails: forward-return HAC *t* = +0.65 … +1.24 across 1–12m (R² ≈ 0.01), placebo p = 0.26, and the post-2009 flicker (*t* = +4.36) is absent pre-2009 (*t* = +0.09). The vol leg is significant (*t* = +9.44) but **mechanical** (VIX ≈ implied vol) and contemporaneous. Uncertainty is a thermometer, not a crystal ball. *Signal is a labelled VIX proxy — the newspaper EPU feed was unreachable in-environment.*
- **Tradability — Mirage.** No uncertainty-timed rule beats buy-and-hold (Sharpe 0.49 / 0.57 vs 0.77); it loses on signal, not costs.